In [2]:
!pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 86.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]


In [5]:
import pandas as pd
from statsmodels.stats.weightstats import ttost_paired


In [19]:

df = pd.read_csv("random_split_runs.csv")

low, upp = -0.02, 0.02  # equivalence margin

results = []

for target, subdf in df.groupby("target"):
    pval, lower, upper = ttost_paired(
        subdf["GAT_auroc"],
        subdf["PPGAT_auroc"],
        low,
        upp
    )
    
    p_low = lower[1]
    p_upp = upper[1]
    
    equivalent = (p_low < 0.05) and (p_upp < 0.05)
    
    results.append({
        "target": target,
        "p_low": p_low,
        "p_upp": p_upp,
        "pvalue": pval,
        "equivalent": equivalent
    })

results_df = pd.DataFrame(results)
print(results_df)

   target     p_low     p_upp    pvalue  equivalent
0  P00915  0.000142  0.000146  0.000146        True
1  P06276  0.000552  0.000734  0.000734        True
2  P08581  0.000386  0.489344  0.489344       False
3  P22303  0.003079  0.073819  0.073819       False
4  P27487  0.000141  0.001770  0.001770        True
5  P34913  0.000201  0.000615  0.000615        True
6  P35968  0.004637  0.287900  0.287900       False
7  P56817  0.015117  0.001631  0.015117        True
8  Q13547  0.000059  0.000254  0.000254        True
9  Q16790  0.000061  0.000101  0.000101        True


In [21]:
import numpy as np
import scipy.stats as st

def paired_ci(x, y, alpha=0.05):
    diff = np.array(x) - np.array(y)
    
    n = len(diff)
    mean = np.mean(diff)
    sem = st.sem(diff)
    
    t_crit = st.t.ppf(1 - alpha/2, df=n-1)
    margin = t_crit * sem
    
    return mean, (mean - margin, mean + margin)



results = []

for target, subdf in df.groupby("target"):
    
    mean, ci = paired_ci(
        subdf["GAT_auroc"],
        subdf["PPGAT_auroc"]
    )
    
    results.append({
        "target": target,
        "mean_diff": mean,
        "ci_low": ci[0],
        "ci_high": ci[1]
    })

ci_df = pd.DataFrame(results)
print(ci_df)

   target  mean_diff    ci_low   ci_high
0  P00915   0.000071 -0.004609  0.004750
1  P06276   0.000747 -0.006117  0.007612
2  P08581   0.019877  0.007854  0.031900
3  P22303   0.009869 -0.005829  0.025567
4  P27487   0.006403  0.000267  0.012539
5  P34913   0.002886 -0.002940  0.008713
6  P35968   0.015420 -0.005481  0.036321
7  P56817  -0.006267 -0.017859  0.005325
8  Q13547   0.003678 -0.000736  0.008092
9  Q16790   0.001281 -0.002718  0.005280


In [25]:
import pandas as pd
import numpy as np
import scipy.stats as st
from statsmodels.stats.weightstats import ttost_paired

# Load data
df = pd.read_csv("random_split_runs.csv")

# equivalence margin
delta = 0.02

def paired_ci(x, y, alpha=0.05):
    diff = np.array(x) - np.array(y)
    n = len(diff)
    mean = np.mean(diff)
    sem = st.sem(diff)
    t_crit = st.t.ppf(1 - alpha/2, df=n-1)
    margin = t_crit * sem
    return mean, (mean - margin, mean + margin)

results = []

for target, subdf in df.groupby("target"):
    
    x = subdf["GAT_auroc"]
    y = subdf["PPGAT_auroc"]
    
    #  TOST 
    pval, lower, upper = ttost_paired(x, y, -delta, delta)
    p_low = lower[1]
    p_upp = upper[1]
    equivalent = (p_low < 0.05) and (p_upp < 0.05)
    
    # CI 
    mean_diff, ci = paired_ci(x, y)

    ci_low, ci_high = ci
    if ci_low > 0:
        direction = "GAT better"
    elif ci_high < 0:
        direction = "PPGAT better"
    else:
        direction = "No clear difference"
    
    # equivalence by CI criterion
    ci_equiv = (ci_low > -delta) and (ci_high < delta)
    
    results.append({
        "target": target, 
        "mean_diff": mean_diff,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_low": p_low,
        "p_upp": p_upp,
        "pval" : pval,
        "tost_equivalent": equivalent,
        "ci_equivalent": ci_equiv,
        "direction": direction
    })

results_df = pd.DataFrame(results)

print(results_df)

   target  mean_diff    ci_low   ci_high     p_low     p_upp      pval  \
0  P00915   0.000071 -0.004609  0.004750  0.000142  0.000146  0.000146   
1  P06276   0.000747 -0.006117  0.007612  0.000552  0.000734  0.000734   
2  P08581   0.019877  0.007854  0.031900  0.000386  0.489344  0.489344   
3  P22303   0.009869 -0.005829  0.025567  0.003079  0.073819  0.073819   
4  P27487   0.006403  0.000267  0.012539  0.000141  0.001770  0.001770   
5  P34913   0.002886 -0.002940  0.008713  0.000201  0.000615  0.000615   
6  P35968   0.015420 -0.005481  0.036321  0.004637  0.287900  0.287900   
7  P56817  -0.006267 -0.017859  0.005325  0.015117  0.001631  0.015117   
8  Q13547   0.003678 -0.000736  0.008092  0.000059  0.000254  0.000254   
9  Q16790   0.001281 -0.002718  0.005280  0.000061  0.000101  0.000101   

   tost_equivalent  ci_equivalent            direction  
0             True           True  No clear difference  
1             True           True  No clear difference  
2            F

In [26]:

# Load data
df = pd.read_csv("analogue_series_runs.csv")

# equivalence margin
delta = 0.02

def paired_ci(x, y, alpha=0.05):
    diff = np.array(x) - np.array(y)
    n = len(diff)
    mean = np.mean(diff)
    sem = st.sem(diff)
    t_crit = st.t.ppf(1 - alpha/2, df=n-1)
    margin = t_crit * sem
    return mean, (mean - margin, mean + margin)

results = []

for target, subdf in df.groupby("target"):
    
    x = subdf["GAT_auroc"]
    y = subdf["PPGAT_auroc"]
    
    #  TOST 
    pval, lower, upper = ttost_paired(x, y, -delta, delta)
    p_low = lower[1]
    p_upp = upper[1]
    equivalent = (p_low < 0.05) and (p_upp < 0.05)
    
    # CI 
    mean_diff, ci = paired_ci(x, y)

    ci_low, ci_high = ci
    if ci_low > 0:
        direction = "GAT better"
    elif ci_high < 0:
        direction = "PPGAT better"
    else:
        direction = "No clear difference"
    
    # equivalence by CI criterion
    ci_equiv = (ci_low > -delta) and (ci_high < delta)
    
    results.append({
        "target": target, 
        "mean_diff": mean_diff,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_low": p_low,
        "p_upp": p_upp,
        "pval" : pval,
        "tost_equivalent": equivalent,
        "ci_equivalent": ci_equiv,
        "direction": direction
    })

results_df = pd.DataFrame(results)

print(results_df)

   target  mean_diff    ci_low   ci_high     p_low     p_upp      pval  \
0  P00915   0.010101  0.001087  0.019115  0.000376  0.019032  0.019032   
1  P06276   0.026154 -0.046331  0.098638  0.075908  0.587383  0.587383   
2  P08581   0.026303  0.000370  0.052237  0.003860  0.731618  0.731618   
3  P22303   0.019723 -0.018075  0.057522  0.021669  0.492383  0.492383   
4  P27487   0.008137 -0.001559  0.017832  0.000644  0.013671  0.013671   
5  P34913   0.032684  0.004282  0.061086  0.003372  0.858617  0.858617   
6  P35968   0.009227 -0.014886  0.033340  0.014084  0.141301  0.141301   
7  P56817  -0.030927 -0.055160 -0.006695  0.860608  0.002150  0.860608   
8  Q13547  -0.090572 -0.210806  0.029662  0.910746  0.031541  0.910746   
9  Q16790   0.001676 -0.015174  0.018525  0.011670  0.019593  0.019593   

   tost_equivalent  ci_equivalent            direction  
0             True           True           GAT better  
1            False          False  No clear difference  
2            F

In [28]:
from scipy.stats import wilcoxon
import pickle

In [29]:
with open('datasets/df_random_vs_st_dataset_revised.pkl', 'rb') as f:
    data = pickle.load(f)

targets = data['accession'].unique()

In [ ]:


# Load data
df = pd.read_csv("random_split_runs.csv")

results = []

for target in targets:
    subset = df[df["target"] == target]
    
    print(subset)
    
    gat = subset["GAT_auroc"]
    gat_rg = subset["GAT_rg_auroc"]
    ppgat = subset["PPGAT_auroc"]

    print(gat)
    

    # Wilcoxon tests 
    p_ppgat_vs_gat = wilcoxon(ppgat, gat).pvalue
    p_gatrg_vs_gat = wilcoxon(gat_rg, gat).pvalue
    p_ppgat_vs_gatrg = wilcoxon(ppgat, gat_rg).pvalue

    results.append({
        "target": target,
        "PPGAT_vs_GAT_p": p_ppgat_vs_gat,
        "GAT_rg_vs_GAT_p": p_gatrg_vs_gat,
        "PPGAT_vs_GAT_rg_p": p_ppgat_vs_gatrg,

    })

results_df = pd.DataFrame(results)

#print(results_df)

   target  run   GAT_acc  GAT_auroc  GAT_rg_acc  GAT_rg_auroc  PPGAT_acc  \
0  P08581    0  0.939929   0.981165    0.922261      0.969537   0.934629   
1  P08581    1  0.925795   0.972203    0.916961      0.968109   0.920495   
2  P08581    2  0.932862   0.980356    0.927562      0.974296   0.876325   
3  P08581    3  0.925795   0.979476    0.939929      0.976597   0.899293   
4  P08581    4  0.959364   0.984723    0.927562      0.979816   0.929329   

   PPGAT_auroc  
0     0.968962  
1     0.962366  
2     0.946017  
3     0.957316  
4     0.963877  
0    0.981165
1    0.972203
2    0.980356
3    0.979476
4    0.984723
Name: GAT_auroc, dtype: float64
